# P3 — Metrics validation

Validate the WRMSSE implementation against hand-computed toy data and the real 12-level aggregation.

Gate requirements:
1. Hand-computed toy panel matches implementation
2. Unit tests pass
3. All 12 levels produced, 42,840 series total
4. All-zeros forecast gives a finite WRMSSE
5. `src/metrics.py` importable; notebook imports rather than redefines

## Setup

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# Confirm src is importable
from src import metrics

# The kernel's cwd is notebooks/, but every data/test path below is written
# relative to the repo root. Anchor them here so they resolve either way.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent

print("Imports successful")
print(f"metrics module: {metrics.__file__}")
print(f"repo root:      {REPO_ROOT}")


Imports successful
metrics module: C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py
repo root:      C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy


## Toy panel — hand-computed validation

Create a small panel (3 series × 10 training days × 3 forecast days) with values chosen to allow hand verification of RMSSE calculation.

### Build toy data

Three series x 10 training days x 3 horizon days, per the P3 brief. Two properties are
deliberate:

- **Forecasts are imperfect.** A perfect forecast makes every expected RMSSE `0.0`, which
  `sqrt(0 / x)` satisfies for any denominator — it couldn't catch a mis-scaled naive baseline.
  See `DECISIONS.md` ("Toy panel rebuilt to actually satisfy validation item 1") for what the
  earlier, perfect-forecast version of this panel looked like and why it was replaced.
- **Days sit at the end of the training period (`d_1904`-`d_1913`).** Weights are defined over
  the final 28 training days (`d_1886`-`d_1913`); a panel outside that range produces no
  weight rows at all. Prices are chosen so the dollar-sales shares come out as round numbers.

Every series also has a non-zero naive denominator, so no RMSSE falls into the degenerate
`naive_mse == 0` branch — that edge case is covered separately in `tests/test_metrics.py`.


In [2]:
# Build toy training data (d_1904-d_1913: 10 days, all inside the weight window)
TOY_TRAIN_DAYS = list(range(1904, 1914))
TOY_HOLDOUT_DAYS = [1914, 1915, 1916]

# Series A: alternating 4/6      -> steps of +/-2
# Series B: linear, step +2      -> steps of +2
# Series C: sparse, leading zeros -> first non-zero on the 3rd day
toy_sales = {
    'A': [4, 6, 4, 6, 4, 6, 4, 6, 4, 6],
    'B': [6, 8, 10, 12, 14, 16, 18, 20, 22, 24],
    'C': [0, 0, 5, 0, 5, 0, 5, 0, 5, 0],
}
# Prices chosen so dollar sales are 100 / 300 / 100 -> weights 0.2 / 0.6 / 0.2
toy_prices = {'A': 2.0, 'B': 2.0, 'C': 5.0}

toy_train = pd.DataFrame({
    'id': [s for s in 'ABC' for _ in TOY_TRAIN_DAYS],
    'item_id': [s for s in 'ABC' for _ in TOY_TRAIN_DAYS],
    'd': TOY_TRAIN_DAYS * 3,
    'sales': toy_sales['A'] + toy_sales['B'] + toy_sales['C'],
    'sell_price': [toy_prices[s] for s in 'ABC' for _ in TOY_TRAIN_DAYS],
})

print("Training data:")
print(toy_train)
print()
print(f"Shape: {toy_train.shape}")
print()
for s in 'ABC':
    dollars = sum(toy_sales[s]) * toy_prices[s]
    print(f"  series {s}: {sum(toy_sales[s]):3d} units x ${toy_prices[s]} = ${dollars:.0f}")


Training data:
   id item_id     d  sales  sell_price
0   A       A  1904      4         2.0
1   A       A  1905      6         2.0
2   A       A  1906      4         2.0
3   A       A  1907      6         2.0
4   A       A  1908      4         2.0
5   A       A  1909      6         2.0
6   A       A  1910      4         2.0
7   A       A  1911      6         2.0
8   A       A  1912      4         2.0
9   A       A  1913      6         2.0
10  B       B  1904      6         2.0
11  B       B  1905      8         2.0
12  B       B  1906     10         2.0
13  B       B  1907     12         2.0
14  B       B  1908     14         2.0
15  B       B  1909     16         2.0
16  B       B  1910     18         2.0
17  B       B  1911     20         2.0
18  B       B  1912     22         2.0
19  B       B  1913     24         2.0
20  C       C  1904      0         5.0
21  C       C  1905      0         5.0
22  C       C  1906      5         5.0
23  C       C  1907      0         5.0
24  C     

In [3]:
# Build toy holdout data (d_1914-d_1916)
# Actuals continue each series' training-period pattern (A: alternating +/-2, B: linear +2,
# C: alternating +/-5) rather than being independently chosen. This keeps the naive one-step
# forecast's holdout error consistent with its training-period naive MSE, which is what
# test_naive_forecast_linear_series and test_one_step_naive_on_random_walk_is_near_one check.
toy_actuals = {'A': [4, 6, 4], 'B': [26, 28, 30], 'C': [5, 0, 5]}

toy_holdout = pd.DataFrame({
    'id': [s for s in 'ABC' for _ in TOY_HOLDOUT_DAYS],
    'item_id': [s for s in 'ABC' for _ in TOY_HOLDOUT_DAYS],
    'd': TOY_HOLDOUT_DAYS * 3,
    'sales': toy_actuals['A'] + toy_actuals['B'] + toy_actuals['C'],
    'sell_price': [toy_prices[s] for s in 'ABC' for _ in TOY_HOLDOUT_DAYS],
})

print("Holdout data:")
print(toy_holdout)
print()
print(f"Shape: {toy_holdout.shape}")


Holdout data:
  id item_id     d  sales  sell_price
0  A       A  1914      4         2.0
1  A       A  1915      6         2.0
2  A       A  1916      4         2.0
3  B       B  1914     26         2.0
4  B       B  1915     28         2.0
5  B       B  1916     30         2.0
6  C       C  1914      5         5.0
7  C       C  1915      0         5.0
8  C       C  1916      5         5.0

Shape: (9, 5)


In [4]:
# Build toy predictions - deliberately IMPERFECT, with a known error per series.
# A: off by -1/+1/-1   B: off by -2 on every day   C: misses the first spike only
toy_preds = {'A': [5, 5, 5], 'B': [24, 26, 28], 'C': [0, 0, 5]}

toy_pred = pd.DataFrame({
    'id': [s for s in 'ABC' for _ in TOY_HOLDOUT_DAYS],
    'd': TOY_HOLDOUT_DAYS * 3,
    'pred': toy_preds['A'] + toy_preds['B'] + toy_preds['C'],
})

print("Predictions:")
print(toy_pred)
print()
for s in 'ABC':
    errs = [a - p for a, p in zip(toy_actuals[s], toy_preds[s])]
    print(f"  series {s}: errors {errs}")


Predictions:
  id     d  pred
0  A  1914     5
1  A  1915     5
2  A  1916     5
3  B  1914    24
4  B  1915    26
5  B  1916    28
6  C  1914     0
7  C  1915     0
8  C  1916     5

  series A: errors [-1, 1, -1]
  series B: errors [2, 2, 2]
  series C: errors [5, 0, 0]


### Hand calculation

Worked on paper; the numbers below are transcribed as literals in the next cell and asserted
against the implementation. Nothing here is computed by re-running the same logic in numpy —
a re-implementation would agree with a shared bug.

**Naive denominator** (mean squared one-step diff, from each series' first non-zero day):

| Series | Training sales | Diffs from first non-zero | Naive MSE |
|---|---|---|---|
| A | 4,6,4,6,4,6,4,6,4,6 | ±2 (nine of them) | mean(4,…,4) = **4** |
| B | 6,8,…,24 | +2 (nine of them) | mean(4,…,4) = **4** |
| C | 0,0,5,0,5,0,5,0,5,0 | from day 3: ±5 (seven) | mean(25,…,25) = **25** |

**Forecast error** over the 3 horizon days:

| Series | Actual | Forecast | Errors | MSE |
|---|---|---|---|---|
| A | 4,6,4 | 5,5,5 | −1,+1,−1 | mean(1,1,1) = **1** |
| B | 26,28,30 | 24,26,28 | −2,−2,−2 | mean(4,4,4) = **4** |
| C | 5,0,5 | 0,0,5 | +5,0,0 | mean(25,0,0) = **25/3** |

**RMSSE = sqrt(MSE / naive MSE):**

- A = sqrt(1 / 4) = **0.5**
- B = sqrt(4 / 4) = **1.0**
- C = sqrt((25/3) / 25) = sqrt(1/3) ≈ **0.57735**

**Weights** — share of dollar sales (units × price) over the weight window, which here is the
whole 10-day training block:

- A: 50 units × \$2 = \$100 &nbsp;|&nbsp; B: 150 × \$2 = \$300 &nbsp;|&nbsp; C: 20 × \$5 = \$100
- Total \$500 → w_A = **0.2**, w_B = **0.6**, w_C = **0.2**  (sum = 1)

**WRMSSE** at the per-item level:

    0.2(0.5) + 0.6(1.0) + 0.2(0.57735) = 0.1 + 0.6 + 0.11547 = 0.81547


In [5]:
# Hand-derived values, transcribed as literals from the markdown above.
# These are the expected values - they are NOT recomputed from the data here, because a
# re-implementation would reproduce any bug the implementation has.
HAND_NAIVE_MSE = {'A': 4.0, 'B': 4.0, 'C': 25.0}
HAND_RMSSE = {'A': 0.5, 'B': 1.0, 'C': (1 / 3) ** 0.5}
HAND_WEIGHTS = {'A': 0.2, 'B': 0.6, 'C': 0.2}
HAND_WRMSSE = 0.2 * 0.5 + 0.6 * 1.0 + 0.2 * (1 / 3) ** 0.5

print("Hand-computed expectations")
print(f"  naive MSE : {HAND_NAIVE_MSE}")
print(f"  RMSSE     : {{'A': 0.5, 'B': 1.0, 'C': {HAND_RMSSE['C']:.6f}}}")
print(f"  weights   : {HAND_WEIGHTS}  (sum = {sum(HAND_WEIGHTS.values())})")
print(f"  WRMSSE    : {HAND_WRMSSE:.6f}")


Hand-computed expectations
  naive MSE : {'A': 4.0, 'B': 4.0, 'C': 25.0}
  RMSSE     : {'A': 0.5, 'B': 1.0, 'C': 0.577350}
  weights   : {'A': 0.2, 'B': 0.6, 'C': 0.2}  (sum = 1.0)
  WRMSSE    : 0.815470


### Compare to implementation

In [6]:
# Call the implementation for each quantity the hand calculation covers.
# level_grouping=[['item_id']] scores one group per series, so the weighted result is
# directly comparable to the hand-computed WRMSSE above.
impl_naive_mse = metrics.compute_naive_baseline_errors(toy_train)
impl_rmsse = metrics.compute_rmsse(toy_holdout, toy_pred, toy_train)
impl_weights = metrics.compute_weights(toy_train)
impl_wrmsse, impl_level_scores = metrics.compute_wrmsse(
    toy_holdout, toy_pred, toy_train, level_grouping=[['item_id']]
)

print("naive MSE:"); print(impl_naive_mse)
print("\nRMSSE:"); print(impl_rmsse)
print("\nweights:"); print(impl_weights)
print(f"\nWRMSSE: {impl_wrmsse}")


naive MSE:
id
A     4.0
B     4.0
C    25.0
Name: naive_mse, dtype: float64

RMSSE:
id
A    0.50000
B    1.00000
C    0.57735
dtype: float64

weights:
id
A    0.2
B    0.6
C    0.2
Name: dollar_sales, dtype: float64

WRMSSE: 0.8154700538379251


In [7]:
# Assert the implementation matches the hand calculation to floating-point tolerance.
TOLERANCE = 1e-9

comparisons = []
for series_id in ['A', 'B', 'C']:
    comparisons.append((f"naive MSE {series_id}", HAND_NAIVE_MSE[series_id], impl_naive_mse[series_id]))
for series_id in ['A', 'B', 'C']:
    comparisons.append((f"RMSSE {series_id}", HAND_RMSSE[series_id], impl_rmsse[series_id]))
for series_id in ['A', 'B', 'C']:
    comparisons.append((f"weight {series_id}", HAND_WEIGHTS[series_id], impl_weights[series_id]))
comparisons.append(("WRMSSE", HAND_WRMSSE, impl_wrmsse))

toy_panel_ok = True
print(f"{'quantity':14s} {'hand':>12s} {'impl':>12s} {'diff':>10s}")
print("-" * 52)
for label, hand_value, impl_value in comparisons:
    diff = abs(float(hand_value) - float(impl_value))
    ok = diff <= TOLERANCE
    toy_panel_ok = toy_panel_ok and ok
    print(f"{label:14s} {float(hand_value):12.6f} {float(impl_value):12.6f} "
          f"{diff:10.2e} {'ok' if ok else 'FAIL'}")

# Weights must also form a valid distribution.
weights_sum_ok = abs(float(impl_weights.sum()) - 1.0) <= TOLERANCE
toy_panel_ok = toy_panel_ok and weights_sum_ok
print(f"\nweights sum to 1: {weights_sum_ok} ({float(impl_weights.sum())})")

# No expected RMSSE is 0 here, so a mis-scaled denominator cannot hide behind sqrt(0/x).
toy_panel_is_trivial = set(HAND_RMSSE.values()) == {0.0}

print()
print(f"{len(comparisons)} hand-computed quantities checked "
      f"(RMSSE and weights, per the P3 brief)")
print("toy panel matches implementation" if toy_panel_ok else "TOY PANEL MISMATCH")

if not toy_panel_ok:
    raise AssertionError("toy panel does not match the hand calculation")


quantity               hand         impl       diff
----------------------------------------------------
naive MSE A        4.000000     4.000000   0.00e+00 ok
naive MSE B        4.000000     4.000000   0.00e+00 ok
naive MSE C       25.000000    25.000000   0.00e+00 ok
RMSSE A            0.500000     0.500000   0.00e+00 ok
RMSSE B            1.000000     1.000000   0.00e+00 ok
RMSSE C            0.577350     0.577350   1.11e-16 ok
weight A           0.200000     0.200000   0.00e+00 ok
weight B           0.600000     0.600000   0.00e+00 ok
weight C           0.200000     0.200000   0.00e+00 ok
WRMSSE             0.815470     0.815470   0.00e+00 ok

weights sum to 1: True (1.0)

10 hand-computed quantities checked (RMSSE and weights, per the P3 brief)
toy panel matches implementation


## Unit tests

`tests/test_metrics.py` is a checked-in source file, not a notebook artifact — this
section only runs it. (It used to be generated from a string literal here, which meant
the notebook silently held its own stale, divergent copy of the tests.)


In [8]:
test_file = REPO_ROOT / "tests" / "test_metrics.py"
assert test_file.exists(), f"missing {test_file} - it is a checked-in file, not generated here"

test_names = [
    line.strip().split("(")[0].replace("def ", "")
    for line in test_file.read_text(encoding="utf-8").splitlines()
    if line.strip().startswith("def test_")
]

print(f"{test_file.relative_to(REPO_ROOT)}: {len(test_names)} tests")
for name in test_names:
    print(f"  {name}")


tests\test_metrics.py: 18 tests
  test_constant_series_zero_error
  test_linear_series_unit_error
  test_all_zeros
  test_sparse_series_starts_from_first_nonzero
  test_int16_sales_do_not_overflow
  test_multiple_series
  test_perfect_forecast_constant_series
  test_naive_forecast_linear_series
  test_multiple_series_separate_scaling
  test_weights_sum_to_one
  test_weights_dollar_basis
  test_weights_respect_date_range
  test_zero_forecast_finite_result_total_only
  test_store_level_perfect_forecast_scores_zero
  test_grouped_levels_all_zero_for_perfect_forecast
  test_weight_window_actually_populated
  test_empty_weight_window_raises
  test_one_step_naive_on_random_walk_is_near_one


In [9]:
import subprocess

# sys.executable is the kernel's own interpreter, so pytest runs in the same
# environment src was installed into - no hardcoded venv path to go stale.
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(test_file), "-v"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)

print("PYTEST OUTPUT:")
print(result.stdout)
if result.stderr:
    print("STDERR:")
    print(result.stderr)
print(f"\nReturn code: {result.returncode}")

# Derive gate inputs from the actual run rather than restating them by hand.
unit_tests_ok = result.returncode == 0
passed_lines = [ln for ln in result.stdout.splitlines() if " PASSED" in ln]
unit_test_count = len(passed_lines)

# Gate item: "non-zero forecast reproduces a hand-summed aggregate" is carried by
# the fan-out tests, so read their result specifically instead of assuming the
# whole suite passing implies this particular check ran.
fanout_lines = [ln for ln in result.stdout.splitlines() if "TestFanOutBugFix" in ln]
fanout_tests_ok = bool(fanout_lines) and all(" PASSED" in ln for ln in fanout_lines)

print(f"\npassed: {unit_test_count}   suite ok: {unit_tests_ok}")
print(f"fan-out / aggregation tests: {len(fanout_lines)} found, ok={fanout_tests_ok}")


PYTEST OUTPUT:
============================= test session starts =============================
platform win32 -- Python 3.11.0, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\venv\Scripts\python.exe
cachedir: .pytest_cache
Fugue tests will be initialized with options:
rootdir: C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy
configfile: pyproject.toml
plugins: anyio-4.14.2, fugue-0.9.7
collecting ... collected 18 items

tests/test_metrics.py::TestComputeNaiveBaselineErrors::test_constant_series_zero_error PASSED [  5%]
tests/test_metrics.py::TestComputeNaiveBaselineErrors::test_linear_series_unit_error PASSED [ 11%]
tests/test_metrics.py::TestComputeNaiveBaselineErrors::test_all_zeros PASSED [ 16%]
tests/test_metrics.py::TestComputeNaiveBaselineErrors::test_sparse_series_starts_from_first_nonzero PASSED [ 22%]
tests/test_metrics.py::TestComputeNaiveBaselineErrors::test_int16_sales_do_not_overflow PASSED [ 27%]
tests/test

## Real data validation

Load real sales data and validate the 12-level aggregation.

In [10]:
import pandas as pd
from pathlib import Path

# Load training data (d_1 through d_1913)
parquet_path = REPO_ROOT / 'data' / 'processed' / 'sales_long.parquet'

if not parquet_path.exists():
    print(f"Warning: {parquet_path} does not exist.")
    print("Real data validation will be skipped.")
    sales_long_train = None
else:
    sales_long_train = pd.read_parquet(
        parquet_path,
        engine='pyarrow',
        filters=[('d', '>=', 1), ('d', '<=', 1913)]
    )
    print(f"✓ Loaded training data: {sales_long_train.shape[0]:,} rows")
    print(f"  Date range: {sales_long_train['date'].min()} to {sales_long_train['date'].max()}")
    print(f"  Unique series (ids): {sales_long_train['id'].nunique():,}")
    print(f"  Columns: {list(sales_long_train.columns)}")

✓ Loaded training data: 46,027,957 rows


  Date range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00


  Unique series (ids): 30,490
  Columns: ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price']


### Count series across all 12 aggregation levels

In [11]:
if sales_long_train is not None:
    # Define the 12 aggregation levels
    levels = [
        ('Total', []),
        ('State', ['state_id']),
        ('Store', ['store_id']),
        ('Category', ['cat_id']),
        ('Department', ['dept_id']),
        ('State×Category', ['state_id', 'cat_id']),
        ('State×Department', ['state_id', 'dept_id']),
        ('Store×Category', ['store_id', 'cat_id']),
        ('Store×Department', ['store_id', 'dept_id']),
        ('Item', ['item_id']),
        ('Item×State', ['item_id', 'state_id']),
        ('Item×Store', ['item_id', 'store_id']),
    ]

    expected_counts = {
        'Total': 1,
        'State': 3,
        'Store': 10,
        'Category': 3,
        'Department': 7,
        'State×Category': 9,
        'State×Department': 21,
        'Store×Category': 30,
        'Store×Department': 70,
        'Item': 3049,
        'Item×State': 9147,
        'Item×Store': 30490,
    }

    print("Series counts by aggregation level:")
    print()
    
    total_series = 0
    all_counts_match = True
    
    for level_name, grouping_cols in levels:
        if grouping_cols:
            # Count unique combinations
            count = sales_long_train[grouping_cols].drop_duplicates().shape[0]
        else:
            # Total level
            count = 1
        
        expected = expected_counts[level_name]
        match = "✓" if count == expected else "✗"
        print(f"{match} {level_name:20s} {count:6,} series (expected {expected:6,})")
        
        if count != expected:
            all_counts_match = False
        
        total_series += count
    
    print()
    print(f"Total series across all levels: {total_series:,}")
    print(f"Expected: 42,840")
    
    levels_ok = total_series == 42840 and all_counts_match
    if levels_ok:
        print("\nAll 12 levels produced 42,840 series (matches expected)")
    else:
        print(f"\nMismatch: got {total_series:,} series, expected 42,840")
else:
    # Flags must exist in both branches, or the gate summary cannot report this item.
    levels_ok = None
    total_series = None
    print("SKIPPED: sales_long.parquet not loaded")


Series counts by aggregation level:

✓ Total                     1 series (expected      1)


✓ State                     3 series (expected      3)


✓ Store                    10 series (expected     10)


✓ Category                  3 series (expected      3)


✓ Department                7 series (expected      7)


...

✓ Store×Category           30 series (expected     30)


✓ Store×Department         70 series (expected     70)


✓ Item                  3,049 series (expected  3,049)


✓ Item×State            9,147 series (expected  9,147)


✓ Item×Store           30,490 series (expected 30,490)

Total series across all levels: 42,840
Expected: 42,840

All 12 levels produced 42,840 series (matches expected)


### Test WRMSSE computation on real data

In [12]:
if sales_long_train is not None:
    # Load holdout data
    sales_long_holdout = pd.read_parquet(
        parquet_path,
        engine='pyarrow',
        filters=[('d', '>=', 1914), ('d', '<=', 1941)]
    )
    print(f"✓ Loaded holdout data: {sales_long_holdout.shape[0]:,} rows")
    print(f"  Date range: {sales_long_holdout['date'].min()} to {sales_long_holdout['date'].max()}")
    print(f"  Unique series: {sales_long_holdout['id'].nunique():,}")

✓ Loaded holdout data: 853,720 rows
  Date range: 2016-04-25 00:00:00 to 2016-05-22 00:00:00
  Unique series: 30,490


In [13]:
# Gate flags default to "did not run" so the summary can tell a skip from a failure.
wrmsse_zero = None
zero_forecast_ok = None
level_scores = None

if sales_long_train is not None and sales_long_holdout is not None:
    zero_pred = sales_long_holdout[['id', 'd']].copy()
    zero_pred['pred'] = 0

    print("Computing WRMSSE with all-zeros forecast (takes several minutes)...")
    print()

    # Deliberately NOT wrapped in try/except. This cell previously caught every
    # exception and printed the traceback, which made a hard failure look like
    # ordinary output and let the gate summary below report a pass.
    wrmsse_zero, level_scores = metrics.compute_wrmsse(
        sales_long_holdout,
        zero_pred,
        sales_long_train,
    )

    print(f"Overall WRMSSE: {wrmsse_zero:.4f}")
    print()
    print("Per-level WRMSSE scores:")
    for level_name, score in level_scores.items():
        print(f"  {level_name:20s} {score:.4f}")

    all_finite = all(np.isfinite(score) for score in level_scores.values())
    zero_forecast_ok = bool(np.isfinite(wrmsse_zero)) and all_finite
    all_twelve = len(level_scores) == 12

    print()
    print(f"WRMSSE finite:            {np.isfinite(wrmsse_zero)}")
    print(f"all per-level finite:     {all_finite}")
    print(f"12 levels returned:       {all_twelve} ({len(level_scores)})")
    if not all_finite:
        for level_name, score in level_scores.items():
            if not np.isfinite(score):
                print(f"  non-finite: {level_name}: {score}")
else:
    print("SKIPPED: real data not loaded")


Computing WRMSSE with all-zeros forecast (takes several minutes)...



C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:448: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  actuals.groupby(grouping_cols + ["d"])


C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:476: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_pred_data.groupby(grouping_cols + ["d"])
C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:498: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  training_actuals.groupby(grouping_cols + ["d"])


C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:448: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  actuals.groupby(grouping_cols + ["d"])


C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:476: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_pred_data.groupby(grouping_cols + ["d"])
C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:498: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  training_actuals.groupby(grouping_cols + ["d"])


...

C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:498: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  training_actuals.groupby(grouping_cols + ["d"])


C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:448: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  actuals.groupby(grouping_cols + ["d"])


C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:476: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  agg_pred_data.groupby(grouping_cols + ["d"])


C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py:498: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  training_actuals.groupby(grouping_cols + ["d"])


Overall WRMSSE: 5.4465

Per-level WRMSSE scores:
  Total                7.5075
  State                6.8836
  Store                6.5295
  Category             7.2139
  Department           7.0274
  State×Category       6.4904
  State×Department     6.2200
  Store×Category       5.9125
  Store×Department     5.4643
  Item                 2.7179
  Item×State           1.9783
  Item×Store           1.4121

WRMSSE finite:            True
all per-level finite:     True
12 levels returned:       True (12)


## Gate checklist summary

In [14]:
from pathlib import Path

# Every box below is derived from a variable produced by an earlier cell. Nothing is
# asserted by hand: a check that did not run shows [?], not [x].
#
# Gate items come from docs/plan/P3-metrics.md.

# "importable; notebook imports rather than redefines" - verify both halves.
metrics_path_ok = Path(metrics.__file__).resolve() == (REPO_ROOT / "src" / "metrics.py").resolve()
redefined = [
    name for name in ("compute_wrmsse", "compute_rmsse", "compute_weights",
                      "compute_naive_baseline_errors")
    if name in globals()
]
import_ok = metrics_path_ok and not redefined


def _fmt(ok):
    if ok is None:
        return "[?]"
    return "[x]" if ok else "[ ]"


checks = [
    ("Hand-computed toy panel matches implementation",
     toy_panel_ok,
     ("all expected values are 0.0 - proves a perfect forecast scores 0, but cannot "
      "detect a mis-scaled denominator")
     if toy_panel_ok and toy_panel_is_trivial else "toy panel RMSSE vs hand calculation"),

    ("Unit tests pass",
     unit_tests_ok,
     f"{unit_test_count} passed via pytest exit code {result.returncode}"),

    ("All 12 levels produced, 42,840 series total",
     levels_ok,
     f"{total_series:,} series across 12 levels" if total_series is not None
     else "skipped: sales_long.parquet not loaded"),

    ("All-zeros forecast gives a finite WRMSSE",
     zero_forecast_ok,
     f"WRMSSE = {wrmsse_zero:.4f}, all {len(level_scores)} level scores finite"
     if zero_forecast_ok else "skipped or failed"),

    ("Non-zero forecast reproduces a hand-summed aggregate (validation 4)",
     fanout_tests_ok,
     "tests/test_metrics.py::TestFanOutBugFix"),

    ("src/metrics.py importable; notebook imports rather than redefines",
     import_ok,
     f"{metrics.__file__}" + (f"; REDEFINED IN NOTEBOOK: {redefined}" if redefined else "")),
]

print("=" * 78)
print("GATE CHECKLIST  (docs/plan/P3-metrics.md)")
print("=" * 78)
print()
for label, ok, detail in checks:
    print(f"- {_fmt(ok)} {label}")
    print(f"      {detail}")
    print()

passed = sum(1 for _, ok, _ in checks if ok is True)
failed = [label for label, ok, _ in checks if ok is False]
notrun = [label for label, ok, _ in checks if ok is None]

print("=" * 78)
print(f"{passed} of {len(checks)} gate items verified")
if failed:
    print(f"FAILED  ({len(failed)}): " + "; ".join(failed))
if notrun:
    print(f"NOT RUN ({len(notrun)}): " + "; ".join(notrun))

if failed or notrun:
    print()
    print("GATE NOT PASSED - do not tick the boxes in P3-metrics.md.")
else:
    print()
    print("All gate items verified in this run.")
    print("Before ticking the boxes: a passing test is not proof the check can fail.")
    print("Confirm each guard goes red against a deliberately broken implementation")
    print("- see DECISIONS.md, 'P3 - metric validation gaps found during implementation'.")
print("=" * 78)


GATE CHECKLIST  (docs/plan/P3-metrics.md)

- [x] Hand-computed toy panel matches implementation
      toy panel RMSSE vs hand calculation

- [x] Unit tests pass
      18 passed via pytest exit code 0

- [x] All 12 levels produced, 42,840 series total
      42,840 series across 12 levels

- [x] All-zeros forecast gives a finite WRMSSE
      WRMSSE = 5.4465, all 12 level scores finite

- [x] Non-zero forecast reproduces a hand-summed aggregate (validation 4)
      tests/test_metrics.py::TestFanOutBugFix

- [x] src/metrics.py importable; notebook imports rather than redefines
      C:\Users\Tirthankar Raha\Documents\GitHub\M5_Forecasting_Accuracy\src\metrics.py

6 of 6 gate items verified

All gate items verified in this run.
Before ticking the boxes: a passing test is not proof the check can fail.
Confirm each guard goes red against a deliberately broken implementation
- see DECISIONS.md, 'P3 - metric validation gaps found during implementation'.
